Imports

In [ ]:
import matplotlib.pyplot as plt

from src.data import load_movielens
from src.content_based import ContentBasedRecommender
from src.popularity import PopularityRecommender
from src.hybrid import HybridRecommender
from src.evaluation import train_test_split_by_time, compare_models

Load Dataset

In [ ]:
ratings, movies, users = load_movielens()

Initialize the Model

In [ ]:
model = ContentBasedRecommender(movies)

Generate Recommendations

In [ ]:
recommendations = model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(recommendations)

Inspect Results

In [ ]:
print(f"Recommendations generated: {len(recommendations)}")
display(recommendations)

## Popularity-Based Recommendations

The baseline recommends movies with at least `min_ratings` ratings and the highest share of positive (>= threshold) ratings. Every user gets the same list, minus movies they've already rated.

In [ ]:
popularity_model = PopularityRecommender(movies, min_ratings=50)

popular_recommendations = popularity_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(popular_recommendations)

## Hybrid Recommendations

Combines both scores: `score = alpha * content_score + (1 - alpha) * popularity_score`. Both scores are rescaled to 0-1 first. `alpha=1` is pure content-based and `alpha=0` is pure popularity.

In [ ]:
hybrid_model = HybridRecommender(movies, alpha=0.5, min_ratings=50)

hybrid_recommendations = hybrid_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(hybrid_recommendations)

## Evaluation

Each user's most recent 20% of ratings are held out as a test set. Each model only sees the training ratings, and we check how many of its top-10 recommendations are movies the user went on to rate 4 or higher.

- **precision@10**: share of the 10 recommendations the user liked
- **recall@10**: share of the user's liked held-out movies that were recommended
- **hit_rate@10**: share of users with at least one liked movie in their top 10
- **coverage**: share of the whole catalog recommended to at least one user

In [ ]:
train, test = train_test_split_by_time(ratings, test_fraction=0.2)

print(f"Train ratings: {len(train):,}")
print(f"Test ratings:  {len(test):,}")

In [ ]:
models = {
    "Content-based": ContentBasedRecommender(movies),
    "Popularity": PopularityRecommender(movies, min_ratings=50),
    "Hybrid (alpha=0.5)": HybridRecommender(movies, alpha=0.5, min_ratings=50),
}

comparison = compare_models(models, train, test, k=10, threshold=4)
display(comparison.round(4))

In [ ]:
metrics = ["precision@10", "recall@10", "hit_rate@10", "coverage"]

fig, axes = plt.subplots(1, len(metrics), figsize=(14, 4))
for ax, metric in zip(axes, metrics):
    comparison[metric].plot.bar(ax=ax)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Recommender comparison")
plt.tight_layout()
plt.show()

### Choosing alpha

Run the hybrid with different weights to see how shifting from popularity (left) to genre similarity (right) changes accuracy and coverage. This takes a minute or two.

In [ ]:
alphas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

sweep = compare_models(
    {a: HybridRecommender(movies, alpha=a, min_ratings=50) for a in alphas},
    train, test, k=10, threshold=4
)
sweep.index.name = "alpha"
display(sweep.round(4))

In [ ]:
fig, axes = plt.subplots(1, len(metrics), figsize=(14, 4))
for ax, metric in zip(axes, metrics):
    ax.plot(sweep.index, sweep[metric], marker="o")
    ax.set_title(metric)
    ax.set_xlabel("alpha")

plt.suptitle("Hybrid performance by alpha (0 = popularity, 1 = content-based)")
plt.tight_layout()
plt.show()

### Results

[Which approach scored highest on precision/recall/hit rate, and which covered more of the catalog? Which alpha worked best, and did the hybrid beat both individual approaches?]